# Rentabilidad, expansión, margen y eficiencia

Modelos Ridge con controles de sucursal/mes y precio log-log por producto. Rentabilidad y apertura combinan pronóstico de ventas, costos operativos estimados y costo estándar de producto. Las asociaciones observacionales no identifican efectos causales ni garantizan rentabilidad.

Las funciones de producción son la única implementación del entrenamiento. Estos experimentos registran artefactos en MLflow; no sustituyen la publicación activa. El DAG publica los siete módulos de forma atómica. No se consultan datos personales de contacto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'hgc-ml').exists())
sys.path.insert(0, str(ROOT / 'hgc-ml'))
from core.settings import postgres_engine
from training.common import setup_tracking
engine = postgres_engine()
setup_tracking()
with engine.connect() as connection:
    load_id = connection.execute(text('SELECT load_id FROM hgc_analytics.loads ORDER BY loaded_at DESC LIMIT 1')).scalar_one()
print('Snapshot:', load_id)


Snapshot: befab29d-54d0-4c88-aee6-0d59c6e120aa


In [2]:
branches=pd.read_sql_table('sys_branch_monthly',engine,schema='hgc_analytics')
products=pd.read_sql_table('sys_product_weekly',engine,schema='hgc_analytics')
from serving.registry import report
from training.economics import train_economics
reports=train_economics(branches,products,report('sales'),load_id)
pd.DataFrame([{'module':name,'rows':len(data['rows']),'algorithm':(data.get('model') or {}).get('algorithm')} for name,data in reports.items()])

2026/10/05 03:55:03 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 03:55:03 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


2026/10/05 03:55:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'hgc_efficiency' already exists. Creating a new version of this model...
Created version '6' of model 'hgc_efficiency'.


2026/10/05 03:55:08 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 03:55:08 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


2026/10/05 03:55:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'hgc_margin' already exists. Creating a new version of this model...
Created version '6' of model 'hgc_margin'.


,module,rows,algorithm
0,profit,144,Ridge · costo logarítmico
1,efficiency,12,Ridge · costo logarítmico
2,expansion,12,weekly_seasonal
3,margin,108,Ridge log-log con efectos de producto y sucursal


In [3]:
for name,data in reports.items():
    print(name, (data.get('model') or {}).get('metrics'), data.get('limitations',''), (data.get('model') or {}).get('limitations',''))

profit {'mae_cost_bob': 1277.7521807074395} 72 meses-sucursal sin costos excluidos; nunca imputados como cero. Referencia estadística de costo operativo; no demuestra desperdicio ni ahorro realizable. Rentabilidad incluye alquiler, electricidad y costo estándar de productos (estimación, no costo contable histórico). No incluye nómina u otros gastos ausentes en bronze. 72 meses-sucursal sin costos excluidos; nunca imputados como cero. Referencia estadística de costo operativo; no demuestra desperdicio ni ahorro realizable. Rentabilidad incluye alquiler, electricidad y costo estándar de productos (estimación, no costo contable histórico). No incluye nómina u otros gastos ausentes en bronze.
efficiency {'mae_cost_bob': 1277.7521807074395}  72 meses-sucursal sin costos excluidos; nunca imputados como cero. Referencia estadística de costo operativo; no demuestra desperdicio ni ahorro realizable. Rentabilidad incluye alquiler, electricidad y costo estándar de productos (estimación, no costo 

In [4]:
from serving.predictors import infer
from serving.schemas import InferenceRequest
row=reports['margin']['rows'][0]
results=[infer('margin',InferenceRequest(id_sucursal=row['id_sucursal'],id_producto=row['id_producto'],price_change=delta)) for delta in (0,.05)]
pd.DataFrame([{key:r[key] for key in ('price','units','value','support')} for r in results])

,price,units,value,support
0,85.00,143.849615,5034.736523,Sin soporte para cambiar precio
1,89.25,143.852270,5646.201614,Sin soporte para cambiar precio
